[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/33_computer_vision/01_images_and_augmentation.ipynb)

# Bài 01 — Ảnh số và data augmentation

**🎯 Mục tiêu**
- Ảnh là tensor: kênh, kiểu dữ liệu, `HWC` vs `CHW`, chuẩn hoá; nội suy khi đổi kích thước; bộ lọc tích chập cổ điển.
- **Data augmentation** cài bằng PyTorch thuần (`affine_grid` + `grid_sample`): lật, dịch, xoay, màu sắc.
- Thí nghiệm: augmentation nào giúp, nào **không** — phải khớp với biến thiên mà dữ liệu train còn thiếu, và phải **giữ nhãn**.
- **Shortcut learning**: mô hình học "màu" thay vì "hình", và augmentation màu phá vỡ đường tắt đó.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/33_computer_vision

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.datasets import SHAPE_CLASSES, make_shapes
from ml_course.visualization import set_style

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
print("thiết bị:", device)

## 1. Ảnh là một tensor

- Ảnh xám: ma trận $H\times W$; ảnh màu: $H\times W\times 3$ (R, G, B). File ảnh thường lưu `uint8` (0–255); mô hình làm việc với `float` (0–1 hoặc đã chuẩn hoá).
- Thư viện ảnh (PIL, OpenCV, matplotlib) dùng **HWC**; PyTorch dùng **CHW** (và batch **NCHW**) — nhầm thứ tự chiều là lỗi kinh điển. OpenCV còn đọc theo thứ tự **BGR**.
- **Chuẩn hoá theo kênh**: $(x - \mu_c)/\sigma_c$ với $\mu, \sigma$ tính trên tập train (hoặc của bộ tiền huấn luyện, vd ImageNet — bài 02).

In [ ]:
X_rgb, y_rgb = make_shapes(6, size=48, rgb=True, seed=4, noise=0.05)
img = X_rgb[0]                                                          # HWC, float trong [0, 1]
img_u8 = (img * 255).round().astype(np.uint8)
chw = torch.tensor(img).permute(2, 0, 1)                                # HWC → CHW
print(f"HWC {img.shape} {img.dtype} | uint8 min/max {img_u8.min()}/{img_u8.max()} | CHW cho PyTorch {tuple(chw.shape)}")

gray = img @ np.array([0.299, 0.587, 0.114])                           # độ sáng theo cảm nhận của mắt (ITU-R BT.601)
fig, axes = plt.subplots(1, 5, figsize=(13, 2.8))
axes[0].imshow(img)
axes[0].set_title("RGB")
for c, (name, cmap) in enumerate([("R", "Reds"), ("G", "Greens"), ("B", "Blues")]):
    axes[c + 1].imshow(img[..., c], cmap=cmap, vmin=0, vmax=1)
    axes[c + 1].set_title(f"kênh {name}")
axes[4].imshow(gray, cmap="gray")
axes[4].set_title("xám = 0.299R + 0.587G + 0.114B")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Đổi kích thước và nội suy

Thu nhỏ/phóng to cần **nội suy**: nearest (giữ cạnh sắc nhưng răng cưa), bilinear (mượt hơn, hơi mờ). Thu nhỏ mạnh mà không làm mờ trước gây **aliasing** (chi tiết nhỏ biến thành hoa văn giả) — `antialias=True`.

In [ ]:
x = torch.tensor(make_shapes(1, size=48, seed=2, noise=0.0)[0])[None]  # (1, 1, 48, 48)
fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
axes[0].imshow(x[0, 0], cmap="gray")
axes[0].set_title("gốc 48×48")
for ax, (mode, kw) in zip(axes[1:], [("nearest", {}), ("bilinear", {"align_corners": False}),
                                     ("bilinear", {"align_corners": False, "antialias": True})]):
    small = F.interpolate(x, size=(12, 12), mode=mode, **kw)
    ax.imshow(small[0, 0], cmap="gray")
    ax.set_title(f"12×12 {mode}" + (" + antialias" if kw.get("antialias") else ""))
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Bộ lọc cổ điển = tích chập với kernel cố định

Trước deep learning, đặc trưng ảnh được **thiết kế tay**: làm mờ Gaussian, phát hiện cạnh Sobel... CNN (chương 29) **học** các kernel này — lớp đầu của CNN train trên ảnh thật thường giống bộ lọc cạnh/màu.

In [ ]:
sobel_x = torch.tensor([[-1.0, 0, 1], [-2, 0, 2], [-1, 0, 1]])
kernels = {"làm mờ 5×5": torch.ones(5, 5) / 25, "Sobel x (cạnh dọc)": sobel_x, "Sobel y (cạnh ngang)": sobel_x.T}
fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
axes[0].imshow(x[0, 0], cmap="gray")
axes[0].set_title("gốc")
for ax, (name, k) in zip(axes[1:], kernels.items()):
    ax.imshow(F.conv2d(x, k[None, None], padding=k.shape[0] // 2)[0, 0], cmap="gray")
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Data augmentation

Augmentation tạo biến thể **giữ nguyên nhãn** của ảnh train ở mỗi lần lấy batch — tương đương "nói" với mô hình: *nhãn không đổi dưới các phép biến đổi này*. Đó là một cách đưa **kiến thức tiên nghiệm** (bất biến) vào mô hình, và một dạng regularization mạnh khi ít dữ liệu.

Cài bằng phép biến đổi affine: ma trận $\theta \in \mathbb R^{2\times 3}$ (xoay, lật, co giãn, dịch) → `affine_grid` sinh lưới toạ độ lấy mẫu → `grid_sample` nội suy. Chạy trên GPU, cả batch một lúc.

In [ ]:
def random_affine(x, g, max_rot=math.pi, max_shift=0.15, flip=True):
    """x: (B, C, H, W). Xoay ngẫu nhiên ±max_rot, dịch ±max_shift (tỉ lệ cạnh ảnh), lật ngang với xác suất 0.5."""
    B = x.shape[0]
    angle = (torch.rand(B, generator=g) * 2 - 1) * max_rot
    shift = (torch.rand(2, B, generator=g) * 2 - 1) * max_shift * 2      # toạ độ chuẩn hoá [-1, 1]
    sx = torch.where(torch.rand(B, generator=g) < 0.5, -1.0, 1.0) if flip else torch.ones(B)
    cos, sin = angle.cos(), angle.sin()
    theta = torch.stack([torch.stack([cos * sx, -sin, shift[0]], 1), torch.stack([sin * sx, cos, shift[1]], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, padding_mode="zeros", align_corners=False)


X_demo = torch.tensor(make_shapes(1, seed=3, noise=0.05)[0][:, None]).repeat(8, 1, 1, 1)
aug_demo = random_affine(X_demo, torch.Generator().manual_seed(0))
fig, axes = plt.subplots(1, 8, figsize=(13, 1.9))
for i, ax in enumerate(axes):
    ax.imshow((X_demo if i == 0 else aug_demo)[i, 0], cmap="gray")
    ax.set_title("gốc" if i == 0 else f"biến thể {i}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Thí nghiệm: chỉ có 300 ảnh train

CNN của chương 29 (conv × 3 + flatten), 1 500 bước Adam, batch 64; test trên 1 000 ảnh; 3 seed cho mỗi cấu hình. Tham chiếu: cùng mô hình train trên 3 000 ảnh, không augmentation.

In [ ]:
def make_cnn(in_ch=1, n_classes=5):
    return nn.Sequential(nn.Conv2d(in_ch, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(64 * 7 * 7, n_classes))


def to_tensor(X):
    return torch.tensor(X[:, None] if X.ndim == 3 else X.transpose(0, 3, 1, 2), dtype=torch.float32)


def train(make_model, X, y, steps=1500, augment=None, seed=0, lr=1e-3):
    torch.manual_seed(seed)                                             # seed TRƯỚC khi khởi tạo trọng số
    model = make_model().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        model.train()
        idx = torch.randint(0, len(X), (64,), generator=g)
        xb = X[idx] if augment is None else augment(X[idx], g)
        opt.zero_grad()
        F.cross_entropy(model(xb.to(device)), y[idx].to(device)).backward()
        opt.step()
    return model


@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    return sum((model(X[i:i + 500].to(device)).argmax(1).cpu() == y[i:i + 500]).sum().item() for i in range(0, len(X), 500)) / len(X)


X300, y300 = make_shapes(300, seed=0)
X_test, y_test = make_shapes(1000, seed=1)
X300, X_test, y300, y_test = to_tensor(X300), to_tensor(X_test), torch.tensor(y300), torch.tensor(y_test)

augmentations = {
    "không augmentation": None,
    "lật ngang": lambda x, g: random_affine(x, g, max_rot=0, max_shift=0, flip=True),
    "dịch ±15%": lambda x, g: random_affine(x, g, max_rot=0, max_shift=0.15, flip=False),
    "xoay + lật + dịch": lambda x, g: random_affine(x, g, max_rot=math.pi, max_shift=0.15, flip=True),
}
aug_results = {}
for name, aug in augmentations.items():
    t0 = time.perf_counter()
    aug_results[name] = [accuracy(train(make_cnn, X300, y300, augment=aug, seed=s), X_test, y_test) for s in range(3)]
    print(f"{name:20s}: accuracy = {np.mean(aug_results[name]):.3f} (các seed: {np.round(aug_results[name], 3)}) | {time.perf_counter() - t0:.0f}s")

X3000, y3000 = make_shapes(3000, seed=0)
ref = accuracy(train(make_cnn, to_tensor(X3000), torch.tensor(y3000)), X_test, y_test)
print(f"{'tham chiếu 3 000 ảnh':20s}: accuracy = {ref:.3f}")

- **Dịch** giúp nhiều nhất: CNN này kết thúc bằng flatten nên **không** bất biến dịch (chương 29), trong khi hình ở ảnh test xuất hiện ở mọi vị trí — với 300 ảnh, nhiều vị trí chưa từng thấy. Augmentation dịch lấp đúng lỗ hổng đó, đưa 300 ảnh tới gần mức của 3 000 ảnh.
- **Lật** cũng giúp (các hình đều đối xứng gương — lật giữ nhãn).
- **Thêm xoay** lại làm mất gần hết lợi ích (ngang mức không augmentation) và rất dao động giữa các seed: các hình trong dữ liệu **đã** xoay ngẫu nhiên, nên xoay không thêm biến thiên mới, chỉ làm bài toán train khó hơn (và nội suy khi xoay làm mờ cạnh) với cùng ngân sách 1 500 bước.

Bài học: augmentation không phải "càng nhiều càng tốt" — chọn phép biến đổi phản ánh **biến thiên có thật ở dữ liệu triển khai mà tập train còn thiếu**, và kiểm chứng bằng validation. Và phép biến đổi phải **giữ nhãn**: lật dọc chữ số 6 thành 9, xoay ảnh X-quang có chữ đánh dấu trái/phải, lật ngang biển báo "rẽ trái"... đều sai.

## 3. Shortcut learning: khi mô hình học "màu" thay vì "hình"

Ảnh màu trong đó, ở tập train, **95% hình có màu đặc trưng theo lớp** (tròn thường đỏ, vuông thường xanh lá...). Khi triển khai, màu là **ngẫu nhiên**. Mô hình có học hình dạng không?

In [ ]:
X_bias, y_bias = make_shapes(2000, seed=5, rgb=True, color_bias=0.95)
X_test_bias, y_test_bias = make_shapes(1000, seed=6, rgb=True, color_bias=0.95)
X_test_rand, y_test_rand = make_shapes(1000, seed=7, rgb=True, color_bias=0.0)
X_bias, X_test_bias, X_test_rand = to_tensor(X_bias), to_tensor(X_test_bias), to_tensor(X_test_rand)
y_bias, y_test_bias, y_test_rand = torch.tensor(y_bias), torch.tensor(y_test_bias), torch.tensor(y_test_rand)

fig, axes = plt.subplots(2, 6, figsize=(12, 4.2))
for j in range(6):
    axes[0, j].imshow(X_bias[j].permute(1, 2, 0))
    axes[0, j].set_title(f"train: {SHAPE_CLASSES[y_bias[j]]}", fontsize=9)
    axes[1, j].imshow(X_test_rand[j].permute(1, 2, 0))
    axes[1, j].set_title(f"triển khai: {SHAPE_CLASSES[y_test_rand[j]]}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def color_augment(x, g):
    """Hoán vị ngẫu nhiên kênh màu, 30% chuyển ảnh xám, nhân độ sáng từng kênh ngẫu nhiên."""
    B = x.shape[0]
    perm = torch.stack([torch.randperm(3, generator=g) for _ in range(B)])
    x = torch.gather(x, 1, perm[:, :, None, None].expand_as(x))
    to_gray = (torch.rand(B, 1, 1, 1, generator=g) < 0.3).float()
    x = to_gray * x.mean(1, keepdim=True).expand_as(x) + (1 - to_gray) * x
    return (x * (0.6 + 0.8 * torch.rand(B, 3, 1, 1, generator=g))).clamp(0, 1)


for name, aug in [("không augmentation", None), ("augmentation màu", color_augment)]:
    accs = np.array([(accuracy(m, X_test_bias, y_test_bias), accuracy(m, X_test_rand, y_test_rand))
                     for m in (train(lambda: make_cnn(3), X_bias, y_bias, augment=aug, seed=s) for s in range(2))]).mean(0)
    print(f"{name:20s}: test cùng phân phối (màu theo lớp) = {accs[0]:.3f} | test triển khai (màu ngẫu nhiên) = {accs[1]:.3f}")

Không augmentation: gần như hoàn hảo trên tập test **cùng phân phối** với train — nhưng sụp đổ khi màu ngẫu nhiên. Mô hình đã chọn **đường tắt** (shortcut): màu dễ học hơn hình dạng và dự đoán đúng 95% thời gian ở tập train. Augmentation màu làm màu **không còn mang thông tin** về nhãn → mô hình buộc phải học hình dạng; độ chính xác trên tập cùng phân phối giảm chút ít nhưng **bền vững** khi triển khai.

Đây là hiện tượng rất thật (Geirhos et al., 2020): mô hình phân loại bệnh từ X-quang học **ký hiệu của bệnh viện** in trên phim, mô hình nhận dạng bò học **đồng cỏ xanh** (bò trên bãi biển thì sai). Điều kiện cần: **tập test phản ánh điều kiện triển khai**, không chỉ là một phần ngẫu nhiên của cùng nguồn dữ liệu train.

## 4. Các kỹ thuật augmentation khác

| Kỹ thuật | Ý tưởng | Dùng khi |
|---|---|---|
| Random resized crop | cắt vùng ngẫu nhiên rồi phóng về kích thước gốc | phân loại ảnh tự nhiên (chuẩn ImageNet) |
| Color jitter | đổi độ sáng, tương phản, bão hoà, sắc độ | điều kiện ánh sáng/máy ảnh khác nhau |
| Cutout / Random erasing | che một vùng ngẫu nhiên | buộc dùng nhiều phần của vật thể, bền với che khuất |
| Mixup / CutMix | trộn hai ảnh (và nhãn) theo tỉ lệ | regularization mạnh, hiệu chỉnh xác suất tốt hơn (chương 37) |
| RandAugment / TrivialAugment | chọn ngẫu nhiên phép biến đổi từ một danh sách | không muốn tinh chỉnh tay từng phép |
| Test-time augmentation | trung bình dự đoán trên vài biến thể ảnh test | tăng nhẹ độ chính xác, tốn thêm thời gian suy luận |

Trong thực tế dùng `torchvision.transforms.v2` (hỗ trợ cả bounding box/mask cho bài 03) hoặc Albumentations. Với detection/segmentation, phép biến đổi hình học phải áp dụng **đồng thời** lên ảnh và nhãn (hộp, mask).

## 🧠 Kiểm tra nhanh

1. HWC và CHW là gì? Vì sao hay nhầm khi chuyển giữa OpenCV/PIL và PyTorch?
2. Vì sao thu nhỏ ảnh cần antialias?
3. Augmentation đưa kiến thức gì vào mô hình? Vì sao thêm xoay lại không giúp trong thí nghiệm?
4. Cho ví dụ phép augmentation **không** giữ nhãn.
5. Shortcut learning là gì? Làm sao phát hiện và giảm thiểu?

## 📌 Tóm tắt

- Ảnh là tensor (C, H, W); chú ý kiểu dữ liệu, thứ tự kênh, chuẩn hoá; đổi kích thước cần nội suy và antialias.
- Augmentation = bất biến tiên nghiệm + regularization; hiệu quả nhất khi lấp đúng biến thiên còn thiếu, phải giữ nhãn và được kiểm chứng.
- Mô hình học đường tắt nếu dữ liệu train cho phép; augmentation và tập test đúng điều kiện triển khai là hai lớp phòng vệ.

## 📚 Tài liệu tham khảo

- Shorten & Khoshgoftaar (2019) — *A survey on Image Data Augmentation for Deep Learning*
- Geirhos et al. (2020) — *Shortcut Learning in Deep Neural Networks*
- Zhang et al. (2018) — *mixup: Beyond Empirical Risk Minimization*; Yun et al. (2019) — *CutMix*
- Cubuk et al. (2020) — *RandAugment*
- Szeliski — *Computer Vision: Algorithms and Applications* (2nd ed.), chương 3